# धडा ०९ - मेटाकॉग्निशन डिझाइन पॅटर्न


## सेटअप

हा नोटबुक Microsoft Agent Framework वापरून Metacognition डिझाइन पॅटर्न दाखवतो.

**पूर्वअट:**
- पर्यावरण चलद्रव्यांच्या माध्यमातून कॉन्फिगर केलेली Azure OpenAI डिप्लॉयमेंट
- Azure CLI प्रमाणीकृत (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## मेटाकॉग्निशन म्हणजे काय?

मेटाकॉग्निशन म्हणजे **विचारांबद्दल विचार करणे**. एआय एजंट्सच्या संदर्भात, याचा अर्थ असे एजंट तयार करणे जे:

- त्यांच्या स्वतःच्या निर्णय आणि विचार प्रक्रियेवर **स्वतःचे परावर्तन** करतात
- **त्रुटी शोधतात** आणि शांतपणे अयशस्वी होण्याऐवजी सुसंगतपणे पुनर्प्राप्त होतात
- त्यांच्या प्रतिसादांची पूर्णता आणि उपयुक्तता **मूल्यमापन करतात**
- जेव्हा प्रारंभिक पद्धत कार्य करत नाही तेव्हा त्यांची धोरणे **अनुकूलित करतात** (उदा., बॅकअप सिस्टमकडे वळणे)

मेटाकॉग्निटिव्ह एजंट केवळ प्रश्नांची उत्तरे देत नाही — तो स्वतःच्या कार्यक्षमतेवर लक्ष ठेवतो आणि त्वरित समायोजन करतो.


## प्राथमिक आणि बॅकअप साधने

एक सामान्य मेटाकॉग्निशन नमुना म्हणजे **फॉलबॅक धोरण**. एजंट प्रथम प्राथमिक साधन वापरतो; जर ते अपयशी ठरले (उदा., 404 त्रुटी), तर एजंट अपयश ओळखतो आणि पारदर्शकपणे बॅकअप साधनावर स्विच करतो.

हे वास्तविक जगातील प्रणालींशी सादृश्य आहे जिथे प्राथमिक सेवा उपलब्ध नसू शकतात आणि एजंट्सना पर्यायी मार्ग निवडण्याआधी स्वतःच समस्या तपासावी लागते.

खाली आम्ही दोन फ्लाइट शोध साधने परिभाषित केली आहेत:
- **प्राथमिक** — पॅरिस, टोकियो, आणि बार्सिलोना या ठिकाणांसाठी
- **बॅकअप** — बर्लिन, सिडनी, आणि न्यू यॉर्क सिटीसाठी


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## त्रुटी पुनर्प्राप्तीसह स्व-प्रतिबिंबित एजंट

खालील एजंटला प्रथम प्राथमिक उड्डाण प्रणालीची चाचणी करण्यासाठी, अयशस्वीपणा ओळखण्यासाठी आणि पारदर्शकपणे बॅकअप प्रणालीकडे परत जाण्यास सूचित केले आहे. प्रत्येक प्रतिक्रियेनंतर तो संक्षेपात स्वतःचे मूल्यमापन करतो की त्याने वापरकर्त्याचा प्रश्न पूर्णपणे उत्तर दिला आहे का.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## स्वमूल्यांकन पद्धत

मेटाकॉग्निशनचा आणखी एक पैलू म्हणजे **स्वमूल्यांकन**: एक स्वतंत्र एजंट (किंवा दुसऱ्या वेळी तेच एजंट) प्रतिसादाचे पूर्णत्व, अचूकता, आणि उपयुक्तता यावर पुनरावलोकन करतो.

खाली आपण `ResponseEvaluator` नावाचा एक एजंट तयार करतो जो प्रवास एजंटांच्या प्रतिसादांचे तीन परिमाणांवर गुणांकन करतो.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## सारांश

या धड्यात आपण Microsoft Agent Framework वापरून **मेटाकॉग्निटिव्ह एजंट्स** कसे तयार करायचे ते शिकलात:

- **स्व-परावर्तन**: एजंट्स जे स्वतःच्या विचारप्रक्रियेवर देखरेख करतात आणि घडलेल्या गोष्टी स्पष्टपणे संवाद साधतात.
- **एरर पुनर्प्राप्ती फॉलबॅकसह**: एक प्राथमिक + बॅकअप टूल पॅटर्न जिथे एजंट अयशस्वी होणे ओळखतो (उदा., 404 त्रुटी) आणि आपोआप पर्यायी स्रोत वापरतो.
- **स्व-मूल्यमापन**: एक वेगळा मूल्यमापन एजंट जो प्रतिसादांच्या संपूर्णता, अचूकता आणि उपयुक्ततेसाठी गुणांकन करतो.

हे पॅटर्न एजंट्सना अधिक मजबूत, स्पष्ट आणि विश्वासार्ह बनवतात — उत्पादन तैनातीसाठी अत्यंत महत्त्वपूर्ण गुणधर्म.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
